# Laboratorio 8: App de Transcripción con Gradio y Whisper

### Bloque: Adquisición de Corpus

**Objetivo:** Diseñar y desplegar una interfaz gráfica (UI) mínima para el modelo de transcripción Whisper, facilitando la interacción entre personas usuarias y el pipeline de adquisición de datos audiovisuales.

**Herramientas principales:** `gradio`, `openai-whisper` 
**Tiempo estimado:** 45 minutos de laboratorio


## Encuadre conceptual - ¿Por qué una interfaz gráfica?

En los laboratorios anteriores trabajamos exclusivamente desde el código. Sin embargo, en el ciclo de vida de un proyecto de PLN, muchas veces necesitamos que personas no técnicas (lingüistas, especialistas en dominio) interactúen con nuestros modelos sin tocar el código fuente. 

**Gradio** nos permite envolver una función de Python dentro de una página web local de forma instantánea. No es solo una cuestión estética: una buena interfaz reduce el error humano al cargar archivos, permite previsualizar resultados de forma amigable y, fundamentalmente, democratiza el acceso a las herramientas de inteligencia artificial que estamos construyendo.

> [!NOTE]
> En este laboratorio conservaremos la decisión de usar Whisper de forma local para que el cambio conceptual sea mínimo: primero aprendiste a descargar y transcribir audio; ahora reutilizas esa lógica dentro de una aplicación interactiva.

---

## Instalación de dependencias

Instalamos Gradio para la interfaz y `openai-whisper` para la transcripción local. 

> [!IMPORTANT]
> Al igual que en el laboratorio 7, la herramienta `ffmpeg` debe estar instalada a nivel sistema para procesar los archivos de audio.

In [1]:
# Instalamos las librerías necesarias de forma silenciosa
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "gradio", "openai-whisper", "-q"], check=True)

CompletedProcess(args=['c:\\Users\\iblis\\Desktop\\NLP\\sanches-sabrina-pln-1c-2026\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'gradio', 'openai-whisper', '-q'], returncode=0)

## Preparación del entorno

Antes de cargar el modelo, verificamos que `ffmpeg` esté disponible. Whisper lo necesita para normalizar y procesar archivos de audio de manera consistente.

In [2]:
import os
import shutil
import gradio as gr
import whisper

def preparar_ffmpeg() -> str:
    """Verifica y configura la ruta de ffmpeg en el sistema."""
    ruta_ffmpeg = os.environ.get("FFMPEG_PATH") or shutil.which("ffmpeg")
    
    if not ruta_ffmpeg or not os.path.exists(ruta_ffmpeg):
        raise FileNotFoundError(
            "No se encontró `ffmpeg` en PATH ni mediante la variable `FFMPEG_PATH`. "
            "Instalalo a nivel sistema y reinicia el kernel. En Windows podés usar `winget install Gyan.FFmpeg`."
        )

    # Aseguramos que la carpeta contenedora esté en el PATH de la sesión actual
    carpeta_ffmpeg = os.path.dirname(ruta_ffmpeg)
    if carpeta_ffmpeg not in os.environ.get("PATH", ""):
        os.environ["PATH"] = carpeta_ffmpeg + os.pathsep + os.environ.get("PATH", "")

    return ruta_ffmpeg

ruta_ffmpeg = preparar_ffmpeg()
print(f"✓ ffmpeg detectado en: {ruta_ffmpeg}")

✓ ffmpeg detectado en: C:\Users\iblis\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1-full_build\bin\ffmpeg.EXE


--- 
## Gestión del Estado: Carga única del modelo

En una aplicación interactiva, **no conviene reinstanciar el modelo en cada consulta**, ya que esto consumiría tiempo y memoria RAM innecesariamente. Por eso lo cargamos una sola vez en el ámbito global (cache) y luego lo reutilizamos.

In [3]:
NOMBRE_MODELO = "small"

# Usamos un patrón de caché simple para evitar recargas accidentales
if "_MODELO_WHISPER_APP" not in globals():
    print(f"Cargando modelo '{NOMBRE_MODELO}' en memoria...")
    _MODELO_WHISPER_APP = whisper.load_model(NOMBRE_MODELO)

modelo = _MODELO_WHISPER_APP
print(f"✓ Modelo listo: whisper-{NOMBRE_MODELO}")

Cargando modelo 'small' en memoria...
✓ Modelo listo: whisper-small


## Lógica del Backend: Función de transcripción

Esta función actúa como el "cerebro" de nuestra app. Recibe la ruta de un archivo de audio (proporcionada por Gradio), ejecuta la inferencia de Whisper y devuelve una cadena de texto estructurada.

In [4]:
def transcribir_audio(audio_file: str) -> str:
    """Recibe un path de audio y devuelve la transcripción formateada."""
    if audio_file is None:
        return "Aviso: No se recibió ningún archivo de audio."

    # Ejecutamos transcripción (forzamos español para este laboratorio)
    resultado = modelo.transcribe(audio_file, language="es", fp16=False, verbose=False)
    
    idioma = resultado.get("language", "desconocido")
    segmentos = len(resultado.get("segments", []))
    texto = resultado.get("text", "").strip()

    # Formateamos la salida para la UI
    encabezado = f"[Idioma detectado: {idioma} | Segmentos: {segmentos}]\n\n"
    return encabezado + (texto if texto else "Error: No se obtuvo texto en la transcripción.")

## Diseño de la interfaz

Utilizamos la clase `gr.Interface` para mapear nuestra función con componentes visuales. Buscamos una estructura limpia donde la entrada y la salida estén claramente identificadas.

In [5]:
# Definimos componentes de entrada y salida con etiquetas claras
audio_input = gr.Audio(sources="upload", type="filepath", label="Subir archivo de audio")
output_text = gr.Textbox(label="Resultado de la Transcripción", lines=12)

# Construimos la aplicación
iface = gr.Interface(
    fn=transcribir_audio,
    inputs=audio_input,
    outputs=output_text,
    title="App de Transcripción Local (Whisper)",
    description=(
        "Cargá un archivo de audio (.mp3, .wav, .m4a) para obtener su transcripción automática. "
        "El procesamiento ocurre íntegramente en esta máquina."
    ),
    flagging_mode="never" # Desactivamos flagging para mantener la UI limpia
)

## Lanzar la aplicación

Al ejecutar la siguiente celda, Gradio levantará un servidor web local. Podés usar la interfaz directamente aquí abajo o abrir la URL que aparecerá en pantalla.

In [6]:
# Lanzamos con inline=True para verlo dentro de la notebook
iface.launch(share=True, show_error=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c18f9f6587cc611a1d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Anexo: Alternativa con la librería `transformers`

Si tu flujo de trabajo ya está integrado con el ecosistema de **Hugging Face**, podés lograr el mismo resultado utilizando la abstracción `pipeline`. Conceptualmente el problema no cambia: el modelo sigue corriendo en local, pero la interfaz de programación (API) es distinta.

In [ ]:
!pip install transformers torch

 30%|██▉       | 35016/117294 [01:48<04:43, 290.31frames/s]

   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   ---------------------------------------  10.2/10.2 MB 57.9 MB/s eta 0:00:01
   ---------------------------------------- 10.2/10.2 MB 49.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


 34%|███▍      | 40416/117294 [02:06<04:18, 297.05frames/s]

In [14]:
import shutil
import tempfile
from pathlib import Path
from transformers import pipeline


def get_pipe_asr():
    global _PIPE_ASR
    if "_PIPE_ASR" not in globals():
        _PIPE_ASR = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-small",
            chunk_length_s=30,
        )
    return _PIPE_ASR


def transcribir_transformers(audio_file: str) -> str:
    if audio_file is None:
        return "Aviso: No hay audio."

    pipe = get_pipe_asr()

    suffix = Path(audio_file).suffix or ".wav"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        temp_path = tmp.name

    shutil.copy2(audio_file, temp_path)

    resultado = pipe(
        temp_path,
        batch_size=8,
        generate_kwargs={"language": "spanish", "task": "transcribe"},
    )
    return resultado["text"]


 39%|███▉      | 45972/117294 [02:23<03:53, 305.00frames/s]

 69%|██████▉   | 81444/117294 [04:03<01:42, 351.42frames/s]

In [15]:
audio_input = gr.Audio(sources="upload", type="filepath", label="Subir audio")
output_text = gr.Textbox(label="Transcripcion", lines=12)

iface = gr.Interface(
    fn=transcribir_transformers,
    inputs=audio_input,
    outputs=output_text,
    title="App de Transcripcion con Transformers",
    description="Subi un audio y obtene la transcripcion con Whisper via transformers.",
    flagging_mode="never",
)

 77%|███████▋  | 90016/117294 [04:29<01:20, 340.12frames/s]

In [1]:
iface.launch(inline=True, show_error=True)

NameError: name 'iface' is not defined

---
# Consolidación y Repaso

## Glosario Acotado
*   **User Interface (UI):** Capa de interacción que media entre la lógica del modelo y la persona usuaria, facilitando el uso de herramientas complejas mediante elementos visuales.
*   **Backend:** En este contexto, refiere a la función de Python que procesa los datos (transcripción) y devuelve una respuesta.
*   **Inferencia Local:** Ejecución de un modelo de IA directamente en el hardware del usuario, preservando la privacidad y sin depender de APIs pagas o conexión a internet externa.
*   **Application State (Estado):** El manejo de variables que persisten entre interacciones (como la instancia del modelo), evitando procesos de carga redundantes.

## Preguntas de Autoevaluación

**1. ¿Cuál es el beneficio de definir el componente `gr.Audio` con `type="filepath"`?**
Permite que Gradio guarde el archivo temporalmente y le pase a nuestra función una ruta de archivo (string). Esto es compatible con lo que espera `whisper.transcribe`, simplificando la integración.

**2. ¿Por qué es una buena práctica técnica cachear el modelo en el diccionario `globals()`?**
Cargar un modelo como Whisper Small puede tomar varios segundos y ocupar gigabytes de RAM. Si lo hiciéramos dentro de la función de transcripción, cada vez que un usuario enviara un audio, la app se congelaría para volver a cargar el modelo, degradando la experiencia y saturando la memoria.

**3. ¿Cómo impacta una interfaz gráfica en la construcción de un corpus?**
Facilita la tarea de curaduría y validación. Un especialista en el tema puede escuchar el audio y leer la transcripción en paralelo, corrigiendo errores de manera mucho más ágil que manipulando archivos de texto sueltos o scripts de terminal.

## Actividades sugeridas

1. **Personalización:** Cambiá el tema visual de la interfaz usando el parámetro `theme` en `gr.Interface` (ej: `theme="soft"` o `theme="monochrome"`). 
2. **Exportación:** Modificá la función de backend para que, además de devolver el texto, guarde un archivo `.txt` con la transcripción en una carpeta local llamada `mis_transcripciones/` cada vez que se ejecute.
3. **Análisis de errores:** Subí un audio con mucho ruido de fondo y observá cómo se comporta la UI. ¿Qué metadata adicional (confianza, timestamps) creés que sería útil mostrarle al usuario en esos casos?

## Recursos y referencias

- [Documentación oficial de Gradio](https://www.gradio.app/docs/)
- [Repositorio de OpenAI Whisper](https://github.com/openai/whisper)
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/)